In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
# Verify file ada
zip_path = '/content/drive/MyDrive/flood_segmentation/dolanan-data-nexus-2026.zip'
print("File exists:", os.path.exists(zip_path))
print("File size:", os.path.getsize(zip_path) / 1e9, "GB")

Mounted at /content/drive
File exists: True
File size: 1.27842612 GB


In [ ]:
import zipfile, os

ZIP_PATH = '/content/drive/MyDrive/flood_segmentation/dolanan-data-nexus-2026.zip'
EXTRACT_PATH = '/content/flood_project'

os.makedirs(EXTRACT_PATH, exist_ok=True)
os.chdir(EXTRACT_PATH)

if not os.path.exists(f'{EXTRACT_PATH}/Dataset_Nexus_DolananData'):
    print("Extracting... (mungkin 5-10 menit)")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)
    print("Done!")
else:
    print("Already extracted, skip.")

print("\nIsi folder:")
for item in os.listdir(EXTRACT_PATH):
    print(f"  - {item}")

Extracting... (mungkin 5-10 menit)
Done!

Isi folder:
  - sample_submission.csv
  - Dataset_Nexus_DolananData


In [ ]:
BASE_PATH = '/content/flood_project/Dataset_Nexus_DolananData'

# Cek semua folder
for split in os.listdir(BASE_PATH):
    split_path = os.path.join(BASE_PATH, split)
    if os.path.isdir(split_path):
        print(f"{split}/")
        for sub in os.listdir(split_path):
            print(f"  - {sub}/")

validation-20260429T073727Z-3-001/
  - validation/
train-20260429T073729Z-3-001/
  - train/
test-20260429T073643Z-3-001/
  - test-20260429T073643Z-3-001/


In [ ]:
    # Cell 1: Setup & Config (FIXED - dengan seed di awal)
    import os
    import json
    import random
    import numpy as np
    from PIL import Image
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    from torch.utils.data.sampler import BatchSampler
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    from collections import Counter
    from tqdm import tqdm

    # ========== SEED UNTUK REPRODUCIBILITY (HARUS DI AWAL) ==========
    def set_seed(seed=42):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        os.environ['PYTHONHASHSEED'] = str(seed)

    set_seed(42)
    print("Reproducibility seeds set to 42")

    # ========== PATHS ==========
    BASE_PATH = "/content/flood_project/Dataset_Nexus_DolananData"

    def find_mask_dir(base):
        train_dir = os.path.join(base, "train-20260429T073729Z-3-001", "train")
        for name in ["masks", "mask"]:
            if os.path.isdir(os.path.join(train_dir, name)):
                return name
        raise FileNotFoundError("mask/masks folder not found")

    MASK_DIR = find_mask_dir(BASE_PATH)

    TRAIN_IMG  = os.path.join(BASE_PATH, "train-20260429T073729Z-3-001", "train", "img")
    TRAIN_MASK = os.path.join(BASE_PATH, "train-20260429T073729Z-3-001", "train", MASK_DIR)

    VAL_IMG  = os.path.join(BASE_PATH, "validation-20260429T073727Z-3-001", "validation", "img")
    VAL_MASK = os.path.join(BASE_PATH, "validation-20260429T073727Z-3-001", "validation", MASK_DIR)

    TEST_IMG = os.path.join(BASE_PATH, "test-20260429T073643Z-3-001", "test-20260429T073643Z-3-001", "test", "img")

    # ========== CLASS DEFINITIONS ==========
    CLASS_NAMES = {
        0: "background", 1: "building flooded", 2: "building non-flooded",
        3: "grass", 4: "pool", 5: "road flooded",
        6: "road non-flooded", 7: "tree", 8: "vehicle", 9: "water"
    }
    NUM_CLASSES = 10
    EMPTY_CLASSES = {2, 6}  # Tidak pernah muncul di dataset

    # Class weights dari EDA (vehicle dan water paling langka)
    CLASS_WEIGHTS = torch.tensor([0.3, 1.0, 0.0, 0.5, 2.5, 0.8, 0.0, 1.2, 15.0, 8.0], dtype=torch.float32)

    # Normalization dari EDA
    MEAN = [0.4195, 0.4572, 0.3500]
    STD  = [0.2045, 0.1901, 0.2066]

    RARE_CLASSES = {4, 8, 9}  # pool, vehicle, water

    print(f"Train images dir: {TRAIN_IMG}")
    print(f"Val images dir:   {VAL_IMG}")
    print(f"Test images dir:  {TEST_IMG}")
    print(f"Mask folder:      {MASK_DIR}")

Reproducibility seeds set to 42
Train images dir: /content/flood_project/Dataset_Nexus_DolananData/train-20260429T073729Z-3-001/train/img
Val images dir:   /content/flood_project/Dataset_Nexus_DolananData/validation-20260429T073727Z-3-001/validation/img
Test images dir:  /content/flood_project/Dataset_Nexus_DolananData/test-20260429T073643Z-3-001/test-20260429T073643Z-3-001/test/img
Mask folder:      masks


In [ ]:
# Cell 2: Lovász-Softmax Loss (FIXED - guard untuk EMPTY_CLASSES)
def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

def lovasz_softmax_flat(probas, labels, classes='present'):
    if classes == 'present':
        # FIXED: Filter EMPTY_CLASSES dari perhitungan loss
        all_present = labels.unique()
        classes = [c.item() for c in all_present if c.item() not in EMPTY_CLASSES]
        if len(classes) == 0:
            return torch.tensor(0.0, device=probas.device)
    loss = 0.0
    for c in classes:
        fg = (labels == c).float()
        if fg.sum() == 0:
            continue
        probas_class = probas[:, c]
        errors = (fg - probas_class).abs()
        errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
        fg_sorted = fg[perm]
        loss += torch.dot(errors_sorted, lovasz_grad(fg_sorted))
    # FIXED: divide by max(1, len(classes)) untuk stabilitas
    return loss / max(len(classes), 1)

def lovasz_softmax(probas, labels, ignore=None):
    probas = F.softmax(probas, dim=1)
    B, C, H, W = probas.shape
    probas = probas.permute(0, 2, 3, 1).reshape(-1, C)
    labels = labels.reshape(-1)
    if ignore is not None:
        valid = labels != ignore
        probas = probas[valid]
        labels = labels[valid]
    return lovasz_softmax_flat(probas, labels, classes='present')

class FloodSegLoss(nn.Module):
    def __init__(self, class_weights, lovasz_weight=0.75):
        super().__init__()
        self.lovasz_weight = lovasz_weight
        self.ce_weight = 1.0 - lovasz_weight
        self.ce_loss = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255, reduction='mean')

    def forward(self, logits, labels):
        ce = self.ce_loss(logits, labels)
        lovasz = lovasz_softmax(logits, labels, ignore=255)
        total = self.lovasz_weight * lovasz + self.ce_weight * ce
        return total, ce, lovasz

loss_fn = FloodSegLoss(CLASS_WEIGHTS, lovasz_weight=0.75)
print(f"Loss: 0.75*Lovász + 0.25*WCE")
print(f"Weights: {CLASS_WEIGHTS.tolist()}")

Loss: 0.75*Lovász + 0.25*WCE
Weights: [0.30000001192092896, 1.0, 0.0, 0.5, 2.5, 0.800000011920929, 0.0, 1.2000000476837158, 15.0, 8.0]


In [ ]:
# Cell 3: Augmentations (sama seperti sebelumnya)
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=0),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussNoise(var_limit=(10, 30), p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

test_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# TTA flip
tta_flip_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=1.0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

print("Augmentations ready.")

Augmentations ready.


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_11459/1585011157.py:8: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 30), p=0.3),


In [ ]:
# Cell 4: Dataset (FIXED - mask validation & cache)
class FloodSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, image_ids=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        all_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith('.jpg')])
        self.ids = image_ids if image_ids is not None else all_ids

        # Cache untuk rare classes (skip tqdm di production)
        self.rare_classes_in_image = {}
        print("Indexing rare classes...")
        for img_id in self.ids:
            mask_path = os.path.join(mask_dir, f"{img_id}.png")
            if os.path.exists(mask_path):
                mask = np.array(Image.open(mask_path))
                # FIXED: Validasi nilai mask
                unique = np.unique(mask)
                invalid = unique[(unique >= NUM_CLASSES) & (unique != 255)]
                if len(invalid) > 0:
                    print(f"WARNING: {img_id} has invalid class values: {invalid}")
                self.rare_classes_in_image[img_id] = set(unique) & RARE_CLASSES
            else:
                self.rare_classes_in_image[img_id] = set()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = np.array(Image.open(os.path.join(self.img_dir, f"{img_id}.jpg")).convert('RGB'))
        mask = np.array(Image.open(os.path.join(self.mask_dir, f"{img_id}.png")))

        # FIXED: Clamp mask ke range yang valid
        mask = np.clip(mask, 0, NUM_CLASSES - 1)

        if self.transform:
            transformed = self.transform(image=img, mask=mask)
            img = transformed['image']
            mask = transformed['mask'].long()
        return img, mask, img_id

    def get_rare_indices(self):
        return [i for i, img_id in enumerate(self.ids) if len(self.rare_classes_in_image.get(img_id, set())) > 0]

    def get_normal_indices(self):
        rare = set(self.get_rare_indices())
        return [i for i in range(len(self.ids)) if i not in rare]

class FloodTestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.ids = sorted([os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith('.jpg')])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = np.array(Image.open(os.path.join(self.img_dir, f"{img_id}.jpg")).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, img_id

print("Dataset classes ready.")

Dataset classes ready.


In [ ]:
# Cell 5: Balanced Batch Sampler (FIXED - infinite loop)
class BalancedBatchSampler(BatchSampler):
    def __init__(self, dataset, batch_size, rare_ratio=0.35, drop_last=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.rare_ratio = rare_ratio
        self.drop_last = drop_last
        self.rare_indices = dataset.get_rare_indices()
        self.normal_indices = dataset.get_normal_indices()
        print(f"BalancedBatchSampler: {len(self.rare_indices)} rare, {len(self.normal_indices)} normal")

    def __iter__(self):
        n_rare = max(1, int(self.batch_size * self.rare_ratio))
        n_normal = self.batch_size - n_rare
        rare_pool = self.rare_indices.copy()
        normal_pool = self.normal_indices.copy()
        random.shuffle(rare_pool)
        random.shuffle(normal_pool)
        i_rare, i_normal = 0, 0

        # FIXED: Batch iteration with length
        n_batches = len(self)
        for _ in range(n_batches):
            batch = []
            for _ in range(n_rare):
                if i_rare >= len(rare_pool):
                    random.shuffle(rare_pool)
                    i_rare = 0
                batch.append(rare_pool[i_rare])
                i_rare += 1
            for _ in range(n_normal):
                if i_normal >= len(normal_pool):
                    random.shuffle(normal_pool)
                    i_normal = 0
                batch.append(normal_pool[i_normal])
                i_normal += 1
            random.shuffle(batch)
            yield batch

        # Handle remaining if not drop_last
        if not self.drop_last and (i_rare < len(rare_pool) or i_normal < len(normal_pool)):
            batch = []
            while i_rare < len(rare_pool) and len(batch) < self.batch_size:
                batch.append(rare_pool[i_rare])
                i_rare += 1
            while i_normal < len(normal_pool) and len(batch) < self.batch_size:
                batch.append(normal_pool[i_normal])
                i_normal += 1
            if len(batch) > 0:
                yield batch

    def __len__(self):
        return len(self.dataset) // self.batch_size

print("Batch sampler ready.")

Batch sampler ready.


In [ ]:
# Cell 6: Create Datasets & DataLoaders (FIXED - hanya satu loader, num_workers guard)
train_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(TRAIN_IMG) if f.endswith('.jpg')])
val_ids   = sorted([os.path.splitext(f)[0] for f in os.listdir(VAL_IMG) if f.endswith('.jpg')])

train_dataset = FloodSegDataset(TRAIN_IMG, TRAIN_MASK, transform=train_transform, image_ids=train_ids)
val_dataset   = FloodSegDataset(VAL_IMG, VAL_MASK, transform=val_transform, image_ids=val_ids)
test_dataset  = FloodTestDataset(TEST_IMG, transform=test_transform)

BATCH_SIZE = 4

# FIXED: num_workers guard untuk Windows
num_workers = 4 if os.name != 'nt' else 0

train_sampler = BalancedBatchSampler(train_dataset, batch_size=BATCH_SIZE, rare_ratio=0.35, drop_last=True)

train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)

print(f"Train: {len(train_dataset)} -> {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} -> {len(val_loader)} batches")
print(f"Test:  {len(test_dataset)} -> {len(test_loader)} batches")

Indexing rare classes...
Indexing rare classes...
BalancedBatchSampler: 927 rare, 517 normal
Train: 1444 -> 361 batches
Val:   448 -> 112 batches
Test:  447 -> 112 batches


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
# Cell 7: Verify Pipeline
def verify_batch(loader, name, num_batches=2):
    print(f"{name}:")
    for i, (imgs, masks, ids) in enumerate(loader):
        if i >= num_batches:
            break
        print(f"  Batch {i}: imgs {imgs.shape}, masks {masks.shape}, range [{imgs.min():.2f}, {imgs.max():.2f}]")
        unique = torch.unique(masks).tolist()
        has_rare = sum(1 for m in masks if len(set(torch.unique(m).tolist()) & RARE_CLASSES) > 0)
        print(f"    Classes: {unique}, rare in batch: {has_rare}/{len(masks)}")

verify_batch(train_loader, "Train (balanced)")
verify_batch(val_loader, "Val")

Train (balanced):


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  Batch 0: imgs torch.Size([4, 3, 512, 512]), masks torch.Size([4, 512, 512]), range [-2.41, 3.15]
    Classes: [0, 1, 3, 4, 5, 7], rare in batch: 1/4
  Batch 1: imgs torch.Size([4, 3, 512, 512]), masks torch.Size([4, 512, 512]), range [-2.41, 3.15]
    Classes: [0, 1, 3, 4, 5, 7], rare in batch: 1/4
Val:
  Batch 0: imgs torch.Size([4, 3, 512, 512]), masks torch.Size([4, 512, 512]), range [-2.38, 3.15]
    Classes: [0, 1, 3, 4, 5, 7, 8], rare in batch: 4/4
  Batch 1: imgs torch.Size([4, 3, 512, 512]), masks torch.Size([4, 512, 512]), range [-2.30, 3.15]
    Classes: [0, 1, 3, 4, 5, 7, 8], rare in batch: 4/4


In [ ]:
# Cell 8: RLE Utilities
def rle_encode(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return ' '.join(str(x) for x in runs)

def mask_to_rle(mask, empty_classes=None):
    if empty_classes is None:
        empty_classes = set()
    result = {}
    for cls_id in range(NUM_CLASSES):
        if cls_id in empty_classes:
            result[cls_id] = ''
        else:
            binary = (mask == cls_id).astype(np.uint8)
            result[cls_id] = rle_encode(binary) if binary.sum() > 0 else ''
    return result

print("RLE utilities ready.")

RLE utilities ready.


In [ ]:
# Cell 9: SegFormer-B5 Model (FIXED - interpolate dinamis)
import torch
from transformers import SegformerForSemanticSegmentation

class SegFormerB3Flood(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super().__init__()
        model_name = "nvidia/mit-b4"

        if pretrained:
            self.model = SegformerForSemanticSegmentation.from_pretrained(
                model_name,
                num_labels=num_classes,
                ignore_mismatched_sizes=True,
                reshape_last_stage=True,
            )
        else:
            from transformers import SegformerConfig
            config = SegformerConfig.from_pretrained(model_name)
            config.num_labels = num_classes
            config.reshape_last_stage = True
            self.model = SegformerForSemanticSegmentation(config)

    def forward(self, pixel_values):
        outputs = self.model(pixel_values=pixel_values)
        logits = outputs.logits  # (B, num_classes, H/4, W/4)

        # FIXED: Dynamic size based on input
        logits = F.interpolate(
            logits,
            size=pixel_values.shape[-2:],  # (H, W) dari input
            mode='bilinear',
            align_corners=False
        )
        return logits

def init_model(device='cuda'):
    model = SegFormerB3Flood(num_classes=NUM_CLASSES, pretrained=True)
    return model.to(device)

print("SegFormer-B3 model ready.")

SegFormer-B3 model ready.


In [ ]:
# Cell 10: Training Configuration & Metrics
from torch.optim import AdamW
import math

class FloodSegMetrics:
    def __init__(self, num_classes, ignore_classes={2, 6}):
        self.num_classes = num_classes
        self.ignore_classes = ignore_classes
        self.reset()

    def reset(self):
        self.confusion_matrix = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)

    def update(self, preds, targets):
        preds = preds.flatten()
        targets = targets.flatten()

        mask = ~np.isin(targets, list(self.ignore_classes))
        preds = preds[mask]
        targets = targets[mask]

        indices = self.num_classes * targets + preds
        m = np.bincount(indices, minlength=self.num_classes ** 2)
        self.confusion_matrix += m.reshape(self.num_classes, self.num_classes)

    def compute_iou(self):
        intersection = np.diag(self.confusion_matrix)
        union = (self.confusion_matrix.sum(axis=1) +
                 self.confusion_matrix.sum(axis=0) -
                 intersection)
        iou = np.zeros(self.num_classes)
        valid = union > 0
        iou[valid] = intersection[valid] / union[valid]
        return iou

    def compute_miou(self):
        ious = self.compute_iou()
        valid_classes = ~np.isin(np.arange(self.num_classes), list(self.ignore_classes))
        return np.mean(ious[valid_classes])

    def compute_dice(self):
        intersection = np.diag(self.confusion_matrix)
        sum_rows = self.confusion_matrix.sum(axis=1)
        sum_cols = self.confusion_matrix.sum(axis=0)
        dice = np.zeros(self.num_classes)
        valid = (sum_rows + sum_cols) > 0
        dice[valid] = 2 * intersection[valid] / (sum_rows[valid] + sum_cols[valid])
        return dice

def configure_optimizer(model, lr=6e-5, weight_decay=0.01):
    """SegFormer recommended: lower LR, higher WD"""
    param_groups = [
        {
            'params': [p for n, p in model.named_parameters()
                      if 'decode_head' in n and p.requires_grad],
            'lr': lr * 2.0,
        },
        {
            'params': [p for n, p in model.named_parameters()
                      if 'decode_head' not in n and p.requires_grad],
            'lr': lr,
        }
    ]
    optimizer = AdamW(param_groups, weight_decay=weight_decay)
    return optimizer

def get_scheduler(optimizer, num_epochs, steps_per_epoch, warmup_epochs=3):
    total_steps = num_epochs * steps_per_epoch
    warmup_steps = warmup_epochs * steps_per_epoch

    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

TRAIN_CONFIG = {
    'num_epochs': 20,
    'lr': 1e-4,
    'weight_decay': 0.01,
    'warmup_epochs': 3,
    'grad_clip': 1.0,
    'save_dir': '/content/drive/MyDrive/flood_segmentation/checkpoints',
    'early_stop_patience': 7,
    'accumulation_steps': 4,
}

os.makedirs(TRAIN_CONFIG['save_dir'], exist_ok=True)
print("Training configuration ready.")

Training configuration ready.


In [ ]:
# Cell 11: Training Loop (FIXED - scheduler per step, bukan per epoch)
from torch.cuda.amp import GradScaler, autocast
import time

class Trainer:
    def __init__(self, model, train_loader, val_loader, loss_fn, optimizer, scheduler,
                 device, config, use_amp=True):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.config = config
        self.use_amp = use_amp and device.type == 'cuda'
        self.scaler = GradScaler() if self.use_amp else None

        self.metrics = FloodSegMetrics(NUM_CLASSES, EMPTY_CLASSES)
        self.best_miou = 0.0
        self.patience_counter = 0
        self.history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'lr': []}

        self.batch_size = BATCH_SIZE
        self.accum_steps = config.get('accumulation_steps', 1)
        self.steps_per_epoch = len(train_loader)

    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        pbar = tqdm(self.train_loader, desc=f'Training E{epoch}')

        self.optimizer.zero_grad()

        for batch_idx, (images, masks, _) in enumerate(pbar):
            images = images.to(self.device)
            masks = masks.to(self.device)

            if self.use_amp:
                with autocast():
                    logits = self.model(images)
                    loss, ce_loss, lovasz_loss = self.loss_fn(logits, masks)
                self.scaler.scale(loss).backward()

                if (batch_idx + 1) % self.accum_steps == 0:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['grad_clip'])
                    self.scaler.step(self.optimizer)
                    # FIXED: Scheduler step di sini (per step, bukan per epoch)
                    self.scheduler.step()
                    self.scaler.update()
                    self.optimizer.zero_grad()
            else:
                logits = self.model(images)
                loss, ce_loss, lovasz_loss = self.loss_fn(logits, masks)
                loss.backward()

                if (batch_idx + 1) % self.accum_steps == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['grad_clip'])
                    self.optimizer.step()
                    # FIXED: Scheduler step di sini
                    self.scheduler.step()
                    self.optimizer.zero_grad()

            total_loss += loss.item()

            # Record current learning rate
            current_lr = self.optimizer.param_groups[0]['lr']

            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'ce': f'{ce_loss.item():.4f}',
                'lov': f'{lovasz_loss.item():.4f}',
                'lr': f'{current_lr:.2e}'
            })

        return total_loss / len(self.train_loader)

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        self.metrics.reset()
        total_loss = 0

        pbar = tqdm(self.val_loader, desc='Validating')
        for images, masks, _ in pbar:
            images = images.to(self.device)
            masks = masks.to(self.device)

            logits = self.model(images)
            loss, _, _ = self.loss_fn(logits, masks)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            self.metrics.update(preds.cpu().numpy(), masks.cpu().numpy())

            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / len(self.val_loader)
        miou = self.metrics.compute_miou()
        ious = self.metrics.compute_iou()
        dice = self.metrics.compute_dice()

        # Log per-class
        print("\n" + "="*65)
        print(f"{'Class':<20} {'IoU':>8} {'Dice':>8} {'Pixels':>12}")
        print("-"*65)
        for c in range(NUM_CLASSES):
            if c in EMPTY_CLASSES:
                continue
            name = CLASS_NAMES[c][:18]
            total_pixels = self.metrics.confusion_matrix[c].sum()
            print(f"{name:<20} {ious[c]:>8.3f} {dice[c]:>8.3f} {total_pixels:>12,}")
        print("-"*65)
        print(f"{'mIoU (avg)':<20} {miou:>8.3f}")
        print("="*65)

        return avg_loss, miou, ious, dice

    def train(self):
        print(f"Starting training for {self.config['num_epochs']} epochs")
        print(f"Device: {self.device}")
        print(f"AMP: {self.use_amp}")
        print(f"Batch size: {self.batch_size}")
        print(f"Grad accumulation: {self.accum_steps}")
        print(f"Steps per epoch: {self.steps_per_epoch}")
        print("="*65)

        for epoch in range(1, self.config['num_epochs'] + 1):
            print(f"\nEpoch {epoch}/{self.config['num_epochs']}")

            start_time = time.time()
            train_loss = self.train_epoch(epoch)

            val_loss, val_miou, ious, dice = self.validate()

            # Save history (scheduler sudah step di train_epoch, tidak perlu di sini)
            current_lr = self.optimizer.param_groups[0]['lr']
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_miou'].append(val_miou)
            self.history['lr'].append(current_lr)

            epoch_time = time.time() - start_time
            print(f"\nEpoch {epoch} completed in {epoch_time:.1f}s")
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | mIoU: {val_miou:.4f} | LR: {current_lr:.2e}")

            checkpoint = {
                'epoch': epoch,
                'model_state_dict': self.model.state_dict(),
                'optimizer_state_dict': self.optimizer.state_dict(),
                'scheduler_state_dict': self.scheduler.state_dict(),
                'best_miou': self.best_miou,
                'history': self.history,
                'val_miou': val_miou,
                'ious': ious,
                'dice': dice,
            }

            torch.save(checkpoint, os.path.join(self.config['save_dir'], 'last_checkpoint.pth'))

            if val_miou > self.best_miou:
                self.best_miou = val_miou
                self.patience_counter = 0
                torch.save(checkpoint, os.path.join(self.config['save_dir'], 'best_model.pth'))
                print(f" New best mIoU: {val_miou:.4f}!")
            else:
                self.patience_counter += 1
                print(f"mIoU did not improve. Patience: {self.patience_counter}/{self.config['early_stop_patience']}")

            if self.patience_counter >= self.config['early_stop_patience']:
                print(f"Early stopping triggered after {epoch} epochs")
                break

        print(f"\nTraining finished! Best mIoU: {self.best_miou:.4f}")

        # Save history ke file JSON agar tidak hilang
        import json
        with open(os.path.join(self.config['save_dir'], 'training_history.json'), 'w') as f:
            # Convert numpy values to Python types
            history_serializable = {k: [float(x) if hasattr(x, 'item') else x for x in v]
                                   for k, v in self.history.items()}
            json.dump(history_serializable, f, indent=2)
        print(f"History saved to {self.config['save_dir']}/training_history.json")

        return self.history

def setup_and_train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    if device.type == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    model = init_model(device)

    optimizer = configure_optimizer(model, lr=TRAIN_CONFIG['lr'], weight_decay=TRAIN_CONFIG['weight_decay'])
    scheduler = get_scheduler(
        optimizer,
        TRAIN_CONFIG['num_epochs'],
        len(train_loader),
        TRAIN_CONFIG['warmup_epochs']
    )

    # Move class weights to device
    loss_fn_device = FloodSegLoss(CLASS_WEIGHTS.to(device), lovasz_weight=0.75)

    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_fn=loss_fn_device,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        config=TRAIN_CONFIG,
        use_amp=True
    )

    history = trainer.train()
    return trainer, history

print("Training pipeline ready. Call setup_and_train() to start training.")

Training pipeline ready. Call setup_and_train() to start training.


In [ ]:
# Cell 12: Quick Test (FIXED - dengan accumulation_steps eksplisit)
def quick_test():
    print("\n=== QUICK TEST RUN (1 EPOCH) ===")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = init_model(device)

    # Use small subset
    small_train = torch.utils.data.Subset(train_dataset, range(32))
    small_val = torch.utils.data.Subset(val_dataset, range(16))

    test_train_loader = DataLoader(small_train, batch_size=4, shuffle=True, num_workers=0)
    test_val_loader = DataLoader(small_val, batch_size=4, shuffle=False, num_workers=0)

    optimizer = configure_optimizer(model, lr=2e-4, weight_decay=0.01)
    scheduler = get_scheduler(optimizer, 1, len(test_train_loader), 0)
    loss_fn_test = FloodSegLoss(CLASS_WEIGHTS.to(device), lovasz_weight=0.75)

    # FIXED: Tambahkan accumulation_steps
    trainer = Trainer(
        model=model,
        train_loader=test_train_loader,
        val_loader=test_val_loader,
        loss_fn=loss_fn_test,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        config={
            'grad_clip': 1.0,
            'early_stop_patience': 10,
            'num_epochs': 1,
            'save_dir': 'test_checkpoints',
            'accumulation_steps': 1  # ← eksplisit
        },
        use_amp=False
    )
    trainer.train()
    print("Quick test completed!")

os.makedirs('test_checkpoints', exist_ok=True)
# Uncomment to test:
quick_test()


=== QUICK TEST RUN (1 EPOCH) ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/246M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/914 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b4
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

Starting training for 1 epochs
Device: cuda
AMP: False
Batch size: 4
Grad accumulation: 1
Steps per epoch: 8

Epoch 1/1



Validating: 100%|██████████| 4/4 [00:01<00:00,  2.52it/s, loss=1.1805]



Class                     IoU     Dice       Pixels
-----------------------------------------------------------------
background              0.156    0.270    1,244,415
building flooded        0.173    0.295      500,620
grass                   0.417    0.589    1,111,283
pool                    0.397    0.568      436,716
road flooded            0.005    0.010      466,921
tree                    0.204    0.338      418,312
vehicle                 0.042    0.081        5,087
water                   0.000    0.000       10,950
-----------------------------------------------------------------
mIoU (avg)              0.174

Epoch 1 completed in 18.0s
Train Loss: 1.1491 | Val Loss: 1.0919 | mIoU: 0.1743 | LR: 0.00e+00
 New best mIoU: 0.1743!

Training finished! Best mIoU: 0.1743
History saved to test_checkpoints/training_history.json
Quick test completed!


In [ ]:
#trainer, history = setup_and_train()

In [ ]:
def resume_training():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = init_model(device)
    optimizer = configure_optimizer(model, lr=TRAIN_CONFIG['lr'])
    scheduler = get_scheduler(optimizer, TRAIN_CONFIG['num_epochs'],
                               len(train_loader), TRAIN_CONFIG['warmup_epochs'])
    loss_fn_device = FloodSegLoss(CLASS_WEIGHTS.to(device), lovasz_weight=0.75)

    last_ckpt = os.path.join(TRAIN_CONFIG['save_dir'], 'last_checkpoint.pth')

    start_epoch = 1
    if os.path.exists(last_ckpt):
        print("Resuming from checkpoint...")
        ckpt = torch.load(last_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_miou = ckpt['best_miou']
        history = ckpt['history']
        print(f"Lanjut dari epoch {ckpt['epoch']}, best mIoU: {best_miou:.4f}")

    trainer = Trainer(model, train_loader, val_loader, loss_fn_device,
                      optimizer, scheduler, device, TRAIN_CONFIG, use_amp=True)
    trainer.best_miou = best_miou
    trainer.history = history

    # Loop manual mulai dari start_epoch
    for epoch in range(start_epoch, TRAIN_CONFIG['num_epochs'] + 1):
        print(f"\nEpoch {epoch}/{TRAIN_CONFIG['num_epochs']}")
        train_loss = trainer.train_epoch(epoch)
        val_loss, val_miou, ious, dice = trainer.validate()

        current_lr = trainer.optimizer.param_groups[0]['lr']
        trainer.history['train_loss'].append(train_loss)
        trainer.history['val_loss'].append(val_loss)
        trainer.history['val_miou'].append(val_miou)
        trainer.history['lr'].append(current_lr)

        print(f"\nEpoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | mIoU: {val_miou:.4f}")

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_miou': trainer.best_miou,
            'history': trainer.history,
            'val_miou': val_miou,
        }
        torch.save(checkpoint, os.path.join(TRAIN_CONFIG['save_dir'], 'last_checkpoint.pth'))

        if val_miou > trainer.best_miou:
            trainer.best_miou = val_miou
            torch.save(checkpoint, os.path.join(TRAIN_CONFIG['save_dir'], 'best_model.pth'))
            print(f"New best mIoU: {val_miou:.4f}!")
        else:
            trainer.patience_counter += 1
            print(f"No improve. Patience: {trainer.patience_counter}/{TRAIN_CONFIG['early_stop_patience']}")

        if trainer.patience_counter >= TRAIN_CONFIG['early_stop_patience']:
            print("Early stopping!")
            break

    return trainer, trainer.history

In [ ]:
trainer, history = resume_training()

Loading weights:   0%|          | 0/914 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b4
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

Resuming from checkpoint...


/tmp/ipykernel_11459/908758609.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if self.use_amp else None


Lanjut dari epoch 19, best mIoU: 0.3693

Epoch 20/20


Training E20:   0%|          | 0/361 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_11459/908758609.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validating: 100%|██████████| 112/112 [00:41<00:00,  2.72it/s, loss=0.8651]



Class                     IoU     Dice       Pixels
-----------------------------------------------------------------
background              0.534    0.696   39,579,459
building flooded        0.357    0.526   15,071,631
grass                   0.551    0.711   29,100,140
pool                    0.305    0.467    6,858,093
road flooded            0.416    0.588   15,625,293
tree                    0.571    0.727   10,740,305
vehicle                 0.148    0.258      106,300
water                   0.035    0.067      359,291
-----------------------------------------------------------------
mIoU (avg)              0.365

Epoch 20 | Train Loss: 0.4735 | Val Loss: 0.8408 | mIoU: 0.3648
No improve. Patience: 1/7


In [ ]:
# Cell 13: Inference dengan TTA (Test Time Augmentation)
import pandas as pd  # FIXED: Import pandas
from PIL import Image as PILImage

@torch.no_grad()
def predict_with_tta(model, image, tta_transform, device, use_flip=True):
    """
    Predict with Test Time Augmentation
    Args:
        image: numpy array (H, W, C) in RGB, values 0-255
    """
    model.eval()

    # Original - langsung transform tanpa denormalize ulang
    orig = tta_transform(image=image)['image'].unsqueeze(0).to(device)
    logits = model(orig)
    probs = F.softmax(logits, dim=1)

    if use_flip:
        # Flipped version
        image_flip = np.fliplr(image).copy()
        flipped = tta_transform(image=image_flip)['image'].unsqueeze(0).to(device)
        logits_flip = model(flipped)
        probs_flip = F.softmax(logits_flip, dim=1)
        # Flip back probabilities
        probs_flip = torch.flip(probs_flip, dims=[-1])
        # Ensemble average
        probs = (probs + probs_flip) / 2

    return torch.argmax(probs, dim=1).squeeze(0).cpu().numpy()

def generate_submission(model, test_img_dir, test_ids, device, use_tta=True,
                        img_size=(640, 480)):
    """
    Generate submission.csv file
    Args:
        test_img_dir: path to test images directory
        test_ids: list of test image IDs (without .jpg)
        device: torch device
        use_tta: whether to use test time augmentation
        img_size: original image size (width, height)
    """
    model.eval()
    results = []

    # Create transform for inference (resize + normalize)
    infer_transform = A.Compose([
        A.Resize(512, 512),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

    print(f"Generating predictions for {len(test_ids)} images...")

    for img_id in tqdm(test_ids, desc="Inference"):
        # FIXED: Load raw image directly from disk
        img_path = os.path.join(test_img_dir, f"{img_id}.jpg")
        img_np = np.array(PILImage.open(img_path).convert('RGB'))

        # Predict
        if use_tta:
            mask = predict_with_tta(
                model, img_np, infer_transform, device, use_flip=True
            )
        else:
            transformed = infer_transform(image=img_np)['image'].unsqueeze(0).to(device)
            logits = model(transformed)
            mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

        # FIXED: Use Image.Resampling.NEAREST (Pillow >= 10.0)
        mask_img = PILImage.fromarray(mask.astype(np.uint8))
        # Check Pillow version for compatibility
        try:
            # Pillow >= 10.0
            mask_img = mask_img.resize(img_size, PILImage.Resampling.NEAREST)
        except AttributeError:
            # Pillow < 10.0 (fallback)
            mask_img = mask_img.resize(img_size, Image.NEAREST)
        mask = np.array(mask_img)

        # Convert to RLE per class
        rles = mask_to_rle(mask, empty_classes=EMPTY_CLASSES)

        for class_id in range(NUM_CLASSES):
            results.append({
                'id': f"{img_id}_{class_id}",
                'encoded_pixels': rles[class_id]
            })

    # Create DataFrame and save
    df = pd.DataFrame(results)
    df.to_csv('submission.csv', index=False)
    print(f"Submission saved! Total rows: {len(df)}")
    print(f"Expected: {len(test_ids)} images × {NUM_CLASSES} classes = {len(test_ids) * NUM_CLASSES}")

    return df

def load_best_model(checkpoint_path, device='cuda'):
    """Load best model from checkpoint"""
    model = init_model(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"Loaded model from epoch {checkpoint['epoch']} with mIoU {checkpoint['best_miou']:.4f}")
    return model

print("Inference utilities ready.")

Inference utilities ready.


In [ ]:
# Cell 14: Visualisasi Prediksi (Sanity Check)
import matplotlib.pyplot as plt

def visualize_predictions(model, test_img_dir, test_ids, device, num_samples=5,
                          save_path=None):
    """
    Visualisasi prediksi pada test set untuk sanity check
    """
    model.eval()

    infer_transform = A.Compose([
        A.Resize(512, 512),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

    # Color mapping untuk visualisasi
    vis_colors = {
        0: [0, 0, 0],        # background - hitam
        1: [255, 0, 0],      # building flooded - merah
        2: [139, 69, 19],    # building non-flooded - coklat
        3: [0, 255, 0],      # grass - hijau
        4: [0, 255, 255],    # pool - cyan
        5: [255, 165, 0],    # road flooded - orange
        6: [128, 128, 128],  # road non-flooded - abu-abu
        7: [0, 100, 0],      # tree - hijau tua
        8: [255, 255, 0],    # vehicle - kuning
        9: [0, 0, 255]       # water - biru
    }

    def mask_to_rgb(mask):
        rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
        for cls, color in vis_colors.items():
            rgb[mask == cls] = color
        return rgb

    samples = np.random.choice(test_ids, min(num_samples, len(test_ids)), replace=False)

    fig, axes = plt.subplots(len(samples), 3, figsize=(15, 4 * len(samples)))
    if len(samples) == 1:
        axes = axes.reshape(1, -1)

    for i, img_id in enumerate(samples):
        # Load image
        img_path = os.path.join(test_img_dir, f"{img_id}.jpg")
        img = np.array(PILImage.open(img_path).convert('RGB'))

        # Predict
        transformed = infer_transform(image=img)['image'].unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(transformed)
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

        # Resize pred back to original
        pred_img = PILImage.fromarray(pred.astype(np.uint8))
        try:
            pred_img = pred_img.resize((640, 480), PILImage.Resampling.NEAREST)
        except AttributeError:
            pred_img = pred_img.resize((640, 480), Image.NEAREST)
        pred_resized = np.array(pred_img)

        # Convert to RGB for visualization
        pred_rgb = mask_to_rgb(pred_resized)

        # Plot
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Original: {img_id}", fontsize=10)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(pred_rgb)
        axes[i, 1].set_title("Prediction", fontsize=10)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(img)
        axes[i, 2].imshow(pred_rgb, alpha=0.5)
        axes[i, 2].set_title("Overlay", fontsize=10)
        axes[i, 2].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved visualization to {save_path}")
    plt.show()

    return samples

print("Visualization utilities ready.")

Visualization utilities ready.


In [ ]:
# Cell 15: Validasi Distribusi Prediksi
def analyze_submission_predictions(submission_path='submission.csv'):
    """
    Analisis distribusi prediksi untuk sanity check
    """
    df = pd.read_csv(submission_path)

    print("="*60)
    print("SUBMISSION PREDICTION ANALYSIS")
    print("="*60)

    # Per-class statistics
    print("\nPer-class detection:")
    print("-"*40)
    for class_id in range(NUM_CLASSES):
        if class_id in EMPTY_CLASSES:
            continue
        class_rows = df[df['id'].str.endswith(f'_{class_id}')]
        non_empty = class_rows[class_rows['encoded_pixels'] != '']
        pct = len(non_empty) / len(class_rows) * 100
        print(f"  {CLASS_NAMES[class_id]:<20}: {len(non_empty):>3}/{len(class_rows)} images ({pct:.1f}%)")

    # Check rare classes
    print("\n🔍 Rare classes check:")
    for class_id in RARE_CLASSES:
        class_rows = df[df['id'].str.endswith(f'_{class_id}')]
        non_empty = class_rows[class_rows['encoded_pixels'] != '']
        if len(non_empty) == 0:
            print(f"  WARNING: {CLASS_NAMES[class_id]} tidak terdeteksi di SATU PUN gambar!")
        elif len(non_empty) < 20:
            print(f"  CAUTION: {CLASS_NAMES[class_id]} hanya terdeteksi di {len(non_empty)} gambar")
        else:
            print(f"  {CLASS_NAMES[class_id]}: terdeteksi di {len(non_empty)} gambar")

    # Sample images with vehicle detection
    vehicle_ids = df[df['id'].str.endswith('_8') & (df['encoded_pixels'] != '')]['id'].str.split('_').str[0].tolist()
    if vehicle_ids:
        print(f"\nContoh gambar dengan vehicle: {vehicle_ids[:10]}")

    # Sample images with water detection
    water_ids = df[df['id'].str.endswith('_9') & (df['encoded_pixels'] != '')]['id'].str.split('_').str[0].tolist()
    if water_ids:
        print(f"Contoh gambar dengan water: {water_ids[:10]}")

    # Total unique images with any non-background prediction
    all_ids = set()
    for class_id in range(NUM_CLASSES):
        if class_id in EMPTY_CLASSES:
            continue
        class_ids = df[df['id'].str.endswith(f'_{class_id}') & (df['encoded_pixels'] != '')]['id'].str.split('_').str[0].tolist()
        all_ids.update(class_ids)

    print(f"\nTotal gambar dengan setidaknya 1 kelas non-background: {len(all_ids)}/{len(df)//10}")

    return df

print("Analysis utilities ready.")

Analysis utilities ready.


In [ ]:
# Cell 16: Validasi Format Submission (FIXED - dengan Pillow version check)
def validate_submission(submission_path='submission.csv'):
    """Validate submission format sesuai aturan kompetisi"""
    df = pd.read_csv(submission_path)

    print("="*60)
    print("SUBMISSION FORMAT VALIDATION")
    print("="*60)

    # Check columns
    assert list(df.columns) == ['id', 'encoded_pixels'], f"Columns: {df.columns}"
    print("Columns: 'id', 'encoded_pixels'")

    # Check 4470 rows (447 images × 10 classes)
    expected_rows = 447 * 10
    assert len(df) == expected_rows, f"Expected {expected_rows} rows, got {len(df)}"
    print(f"Total rows: {len(df)} (expected {expected_rows})")

    # Check ID format
    sample_id = df['id'].iloc[0]
    assert '_' in sample_id, f"ID format error: {sample_id}"
    print(f"ID format: '{sample_id}' (image_class)")

    # Check each image has exactly 10 classes
    img_ids = set([x.split('_')[0] for x in df['id']])
    expected_img_count = 447
    assert len(img_ids) == expected_img_count, f"Expected {expected_img_count} images, got {len(img_ids)}"

    for img_id in list(img_ids)[:5]:  # Check first 5 as sample
        class_ids = [int(x.split('_')[1]) for x in df['id'] if x.startswith(f"{img_id}_")]
        assert len(class_ids) == 10, f"Image {img_id} has {len(class_ids)} classes, expected 10"
        assert set(class_ids) == set(range(10)), f"Image {img_id} missing classes: {set(range(10)) - set(class_ids)}"
    print(f"Each of {len(img_ids)} images has exactly 10 classes (0-9)")

    # Check RLE format
    invalid_rows = []
    for idx, row in df.iterrows():
        if pd.notna(row['encoded_pixels']) and row['encoded_pixels'] != '':
            parts = row['encoded_pixels'].split()
            if len(parts) % 2 != 0:
                invalid_rows.append(idx)

    if invalid_rows:
        print(f"RLE format error at rows: {invalid_rows}")
    else:
        print("All RLE encodings have valid format (even number of integers)")

    # Check empty classes (2 and 6)
    for class_id in EMPTY_CLASSES:
        class_rows = df[df['id'].str.endswith(f'_{class_id}')]
        non_empty = class_rows[class_rows['encoded_pixels'] != '']
        if len(non_empty) > 0:
            print(f"WARNING: Class {class_id} ({CLASS_NAMES[class_id]}) has {len(non_empty)} non-empty predictions")
        else:
            print(f"Class {class_id} ({CLASS_NAMES[class_id]}): all empty (correct)")

    print("\n" + "="*60)
    print("SUBMISSION FORMAT IS VALID!")
    print("="*60)
    return True

print("Validation utilities ready.")

Validation utilities ready.


In [ ]:
def resume_training():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = init_model(device)
    optimizer = configure_optimizer(model, lr=TRAIN_CONFIG['lr'])
    scheduler = get_scheduler(optimizer, TRAIN_CONFIG['num_epochs'],
                               len(train_loader), TRAIN_CONFIG['warmup_epochs'])
    loss_fn_device = FloodSegLoss(CLASS_WEIGHTS.to(device), lovasz_weight=0.75)

    last_ckpt = os.path.join(TRAIN_CONFIG['save_dir'], 'last_checkpoint.pth')

    start_epoch = 1
    if os.path.exists(last_ckpt):
        print("Resuming from checkpoint...")
        ckpt = torch.load(last_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_miou = ckpt['best_miou']
        history = ckpt['history']
        print(f"Lanjut dari epoch {ckpt['epoch']}, best mIoU: {best_miou:.4f}")

    trainer = Trainer(model, train_loader, val_loader, loss_fn_device,
                      optimizer, scheduler, device, TRAIN_CONFIG, use_amp=True)
    trainer.best_miou = best_miou
    trainer.history = history

    # Loop manual mulai dari start_epoch
    for epoch in range(start_epoch, TRAIN_CONFIG['num_epochs'] + 1):
        print(f"\nEpoch {epoch}/{TRAIN_CONFIG['num_epochs']}")
        train_loss = trainer.train_epoch(epoch)
        val_loss, val_miou, ious, dice = trainer.validate()

        current_lr = trainer.optimizer.param_groups[0]['lr']
        trainer.history['train_loss'].append(train_loss)
        trainer.history['val_loss'].append(val_loss)
        trainer.history['val_miou'].append(val_miou)
        trainer.history['lr'].append(current_lr)

        print(f"\nEpoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | mIoU: {val_miou:.4f}")

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_miou': trainer.best_miou,
            'history': trainer.history,
            'val_miou': val_miou,
        }
        torch.save(checkpoint, os.path.join(TRAIN_CONFIG['save_dir'], 'last_checkpoint.pth'))

        if val_miou > trainer.best_miou:
            trainer.best_miou = val_miou
            torch.save(checkpoint, os.path.join(TRAIN_CONFIG['save_dir'], 'best_model.pth'))
            print(f"New best mIoU: {val_miou:.4f}!")
        else:
            trainer.patience_counter += 1
            print(f"No improve. Patience: {trainer.patience_counter}/{TRAIN_CONFIG['early_stop_patience']}")

        if trainer.patience_counter >= TRAIN_CONFIG['early_stop_patience']:
            print("Early stopping!")
            break

    return trainer, trainer.history

In [ ]:
trainer, history = resume_training()

Loading weights:   0%|          | 0/914 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b4
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

Resuming from checkpoint...
Lanjut dari epoch 20, best mIoU: 0.3693


/tmp/ipykernel_11459/908758609.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if self.use_amp else None


In [ ]:
device = torch.device('cuda')

model = load_best_model(
    '/content/drive/MyDrive/flood_segmentation/checkpoints/best_model.pth',
    device=device
)
test_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(TEST_IMG) if f.endswith('.jpg')])
print(f"Test images: {len(test_ids)}")
df = generate_submission(model, TEST_IMG, test_ids, device, use_tta=True)

Loading weights:   0%|          | 0/914 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b4
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

Loaded model from epoch 19 with mIoU 0.3693
Test images: 447
Generating predictions for 447 images...


Inference: 100%|██████████| 447/447 [01:28<00:00,  5.07it/s]


Submission saved! Total rows: 4470
Expected: 447 images × 10 classes = 4470


In [ ]:
validate_submission('submission.csv')

SUBMISSION FORMAT VALIDATION
Columns: 'id', 'encoded_pixels'
Total rows: 4470 (expected 4470)
ID format: '10163_0' (image_class)
Each of 447 images has exactly 10 classes (0-9)
All RLE encodings have valid format (even number of integers)

SUBMISSION FORMAT IS VALID!


True

In [ ]:
from google.colab import files
files.download('/content/flood_project/submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

history_data = {
    'run': 'run1',
    'config': TRAIN_CONFIG,
    'best_miou': trainer.best_miou,
    'history': trainer.history
}

with open('/content/drive/MyDrive/flood_segmentation/training_history_run1.json', 'w') as f:
    json.dump(history_data, f, indent=2)
print("History saved!")

History saved!
